In [5]:
import sys
import os
import  glob
import pandas as pd
import matplotlib.pyplot as plt
import  seaborn as sns


In [ ]:
sys.path.append(os.path.abspath(""))
from ingest import fetch_metro_sites, fetch_tirtl_traffic

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (14, 6)

PARQUET_PATH = "data/phase1_baseline.parquet"
HISTORICAL_DIR = "data/historical_traffic"

BASELINE_START = "01-11-2025"
BASELINE_END = "03-03-2026"

if os.path.exists(PARQUET_PATH):
    print(print(f"Loading existing combined baseline data from {PARQUET_PATH}..."))
    df = pd.read_parquet(PARQUET_PATH)

else:
    print("Building hybrid baseline dataset from daily CSVs...")
    metro_sites = fetch_metro_sites("data/tirtl_sites.csv")
    valid_sites_str = [str(s) for s in metro_sites]

    # 1. Find all csv files oin the folder 
    all_files = glob.glob(os.path.join(HISTORICAL_DIR, "*.csv"))
    print(f"Found {len(all_files)} daily CSV files. Processing...")

    df_list = []

    # 2. Loop through each file, filter it and save the chunk
    for file in all_files:
        temp_df = pd.read_csv(file)

        # inconsitencies with cols name
        if "site" in temp_df.columns:
            temp_df = temp_df.rename(columns = {"site":"site_id"})
        if "vehicleclass" in temp_df.columns:
            temp_df = temp_df.rename(columns={"vehicleclass" : "vehicle_class"})

        temp_df["date"] = pd.to_datetime(temp_df["date"])

        # Filter historical data for the required sites classes and baseline dates
        mask = (
            (temp_df["site_id"].astype(str).isin(valid_sites_str)) &
            (temp_df["vehicle_class"].astype(str).isin(["1", "5", "6", "7"])) &
            (temp_df["date"] >= BASELINE_START) &
            (temp_df["date"] <= BASELINE_END)
        )

        filtered_chunk = temp_df.loc[mask].copy()

        # Only append if the chunk actually has data
        if not filtered_chunk.empty:
            # Memory optimization
            filtered_chunk["site_id"] = filtered_chunk["site_id"].astype("category")
            filtered_chunk["heading"] = filtered_chunk["heading"].astype("category")
            filtered_chunk["speed_bin"] = filtered_chunk["speed_bin"].astype("category")
            filtered_chunk["vehicle_class"] = filtered_chunk["vehicle_class"].astype("category")
            filtered_chunk["time_bin"] = filtered_chunk["time_bin"].astype("int16")
            filtered_chunk["volume"] = filtered_chunk["volume"].astype("uint16")
            # Memory optimization: Convert "HH:MM" to a 15-min bin index (0 to 95)
            hours = filtered_chunk["time_bin"].astype(str).str.split(':').str[0].astype("int8")
            mins = filtered_chunk["time_bin"].astype(str).str.split(':').str[1].astype("int8")
            
            # Formula: (Hour * 4) + (Minutes / 15)
            filtered_chunk["time_bin"] = (hours * 4 + (mins // 15)).astype("int8")
            df_list.append(filtered_chunk)

    # 3. Combine all the filtred daily chunks into one master df
    if len(df_list) == 0:
        raise ValueError("Files were found, but after filtering for dates/sites/classes, exactly 0 rows were left. Check your date ranges!")
    
    print("Concatenating filtered chunks...")
    df = pd.concat(df_list, ignore_index=True)
    
    
    # 4.  Save clean and combined  dataset to parquet
    df.to_parquet(PARQUET_PATH, index=False)
    print(f"Saved complete baseline data (NOV 1 - Mar 3) to {PARQUET_PATH}")


2026-04-08 14:09:56,688 - INFO - Filtering TIRTL sites by boundary
2026-04-08 14:09:56,701 - INFO - Retained 379 active sites within metro bounding box.


Building hybrid baseline dataset from daily CSVs...
Found 120 daily CSV files. Processing...
Concatenating filtered chunks...
Saved complete baseline data (NOV 1 - Mar 3) to data/phase1_baseline.parquet


In [25]:
print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
display(df.head())

Dataset shape: (46562659, 7)
Date range: 2025-11-01 to 2026-02-28


,date,time_bin,site_id,heading,vehicle_class,speed_bin,volume
0,2025-11-01,0:00,2,W,1,70km/hr to < 75km/hr,2
1,2025-11-01,0:00,2,W,1,75km/hr to < 80km/hr,1
2,2025-11-01,0:00,2,W,1,80km/hr to < 85km/hr,4
3,2025-11-01,0:00,2,W,1,85km/hr to < 90km/hr,8
4,2025-11-01,0:00,2,W,5,85km/hr to < 90km/hr,1


In [29]:
(temp_df['date'].dt.time != pd.to_datetime("00:00:00").time()).any()

np.False_